# Executor Failures: Node Eviction, Heartbeats, and State Handling

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 02_executor_failure.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: What happens when an executor process crashes: heartbeat timeout, cluster manager notification, task rescheduling, and shuffle data loss.
- **Why does this topic exist?**: Distributed systems must be designed for frequent hardware and container failures.
- **What problem does it solve?**: Understanding why shuffle files may force upstream stage recomputation.
- **Where is it used in real-world Data Engineering?**: Running Spark on AWS EC2 Spot Instances.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Cluster architecture, lineage, shuffle |
| **Required depth** | Working knowledge: can explain executor failure detection and recovery |
| **Data Engineer** | Diagnose executor loss and its impact on running jobs. |
| **Senior Data Engineer** | Design for spot instances, node drains, and shuffle loss. |
| **Spark specialist** | Reason about heartbeat timeouts, shuffle services, and dynamic allocation. |
| **Next topic** | `11_fault_tolerance_and_reliability/03_task_retries.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Executors are disposable, but their state is not free to lose. When an executor dies, its running tasks are
resubmitted and its cached blocks disappear; shuffle outputs may also be lost unless an external shuffle service
retains them. The driver usually survives and the job continues, but repeated executor loss can turn a healthy job
into a long recomputation loop.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Heartbeat** | Liveness signal | Executors periodically report to the driver |
| **Network timeout** | Failure detection window | `spark.network.timeout` default 120s |
| **Executor loss** | Process/container death | Tasks rerun; cached blocks lost |
| **Shuffle loss** | Map outputs unavailable | May trigger parent-stage recomputation |
| **External shuffle service** | Retains map outputs | Useful for dynamic allocation and ephemeral executors |
| **Dynamic allocation** | Scale executors | Removes idle executors; interacts with shuffle retention |

### How it works, step by step

1. An executor stops sending heartbeats or exits.
2. The driver or cluster manager marks it dead after the timeout.
3. Running tasks are marked failed and resubmitted elsewhere.
4. Cached blocks on that executor are lost and must be recomputed.
5. Shuffle map outputs may be lost, forcing a parent stage to rerun.
6. The cluster manager may request a replacement executor.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Executor Failure Lifecycle
1. Heartbeat stops (`spark.network.timeout` = 120s).
2. Master marks executor dead; requests replacement.
3. Running tasks rescheduled on healthy workers.
4. Shuffle files lost -> Upstream ShuffleMapStage recomputed.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Executor_Failure_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
conf = spark.sparkContext.getConf()
print("spark.network.timeout:         ", conf.get("spark.network.timeout", "120s"))
print("spark.executor.heartbeatInterval: ", conf.get("spark.executor.heartbeatInterval", "10s"))

---

## Experiment: External Shuffle Service Resilience

- **1. What are we testing?**: Explain role of External Shuffle Service during executor loss.
- **2. What do we expect?**: ESS preserves shuffle files after executor crash.

In [ ]:
print('ESS runs as an auxiliary node daemon so shuffle files survive executor JVM crashes.')

**3. What happened?**: Explained ESS architecture.  
**4. Why did it happen?**: ESS decouples shuffle storage from executor process lifecycle.  
**5. Practical Engineering Lesson**: Enable ESS when running dynamic allocation or preemptible instances.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Spot-instance churn

- **Industry**: Cloud SaaS
- **Scenario**: A Spark job runs on spot instances and loses executors during price spikes.
- **Problem**: Each loss forces task retries and may recompute shuffle stages.
- **Why this concept is relevant**: An external shuffle service and checkpointing reduce the cost of losing an executor.
- **Implementation idea**: Enable the shuffle service, use dynamic allocation carefully, and monitor executor loss rate.
- **Expected result**: A job that tolerates churn with bounded slowdown.
- **Alternative**: Use on-demand instances for the critical stage.
- **Trade-offs**: On-demand instances are more reliable but cost more; a hybrid pool balances cost and stability.

### Use Case 2: Node drain during maintenance

- **Industry**: Telecom
- **Scenario**: A cluster operator drains a node while a long ETL job is running.
- **Problem**: Executors on the node disappear and the job stalls or recomputes.
- **Why this concept is relevant**: Graceful decommissioning and shuffle retention reduce the impact of planned maintenance.
- **Implementation idea**: Use decommissioning support, drain nodes during low-traffic windows, and monitor stage retries.
- **Expected result**: Maintenance without a failed SLA.
- **Alternative**: Pause the job during maintenance.
- **Trade-offs**: Pausing is simple but delays delivery; decommissioning keeps the job running.

---

## Business Value

- Higher effective availability on unreliable infrastructure.
- Lower cost from spot-instance tolerance.
- Faster recovery from node failures.
- Clear operational procedures for maintenance.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Monitor executor loss rate and stage retries.
- Use an external shuffle service when executors are ephemeral.
- Avoid caching critical data only in executor memory.
- Tune heartbeat/timeout settings carefully; too aggressive causes false failures.
- Plan for driver failure separately; executor recovery does not cover the driver.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Observe executor loss

**Steps**

1. Run a job with a cached DataFrame and a shuffle.
2. Stop one executor from the cluster UI or command line.
3. Watch task retries, cache loss, and stage recomputation in the Spark UI.

**What to observe**: Tasks rerun, cached blocks disappear, and shuffle loss may trigger a parent stage.

**Healthy result**: The job completes with bounded extra work.

**Problematic result**: The job loops because executors keep failing.

**Engineering takeaway**: Executor loss is expected; design for its cost.

---

## Trade-offs and Alternatives

### Advantages

- Automatic task recovery
- Works with elastic clusters
- Supports spot instances

### Limitations

- Cache and shuffle loss
- Repeated failures can loop
- Driver failure is still fatal

### When to use it

- Cloud clusters
- Spot instances
- Node maintenance

### When not to use it

- Relying on executor-local state
- Ignoring shuffle loss
- Very aggressive timeouts

### Alternatives inside Spark

- Heartbeats
- Dynamic allocation
- External shuffle service

### Alternatives outside Spark

- Cluster manager health checks
- Node decommissioning
- Multi-AZ storage

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Heartbeat Interval
**Problem Statement**: Print default heartbeat interval.  
**Input / Context**: conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Stage Invalidation
**Problem Statement**: Why lost shuffle files invalidate stages.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Dynamic Allocation with ESS
**Problem Statement**: Why dynamic allocation requires ESS.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Executors keep disappearing

**Scenario**: A job loses executors every few minutes but the application logs show no task exceptions.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check cluster manager events for preemption, eviction, or node drains.
2. Compare container memory limits with executor usage.
3. Look for heartbeat timeouts caused by network or GC pauses.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Heartbeat Interval):
```python
print(spark.conf.get('spark.executor.heartbeatInterval','10s'))
```

#### Solution 2 (Stage Invalidation):
```python
print('Reducers cannot fetch missing shuffle map files; parent stage must re-run.')
```

#### Solution 3 (Dynamic Allocation with ESS):
```python
print('Without ESS, deallocated executors would delete their shuffle files.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Executor Crash**: Driver automatically reschedules tasks and recomputes lost data.
- **Shuffle Loss**: Forces upstream stage rerun unless ESS is used.

### Essential Technical Terms
- `Heartbeat`: Periodic health ping from executor to driver.
- `ESS`: External Shuffle Service.

### Common Engineering Mistakes
- **Assuming application dies with executor**: Spark applications survive executor crashes gracefully.

### Interview Questions on This Topic
- **Q: What happens if an executor holding shuffle files crashes?**
  - *A*: The DAGScheduler invalidates the upstream ShuffleMapStage and reschedules the lost map tasks to regenerate the shuffle files.


### More Interview Questions

- **Q: What happens to cached data when an executor dies?**
  - *A*: Blocks cached on that executor are lost; subsequent accesses recompute them from lineage or read them from another replica if the storage level has replication.
- **Q: Why does executor loss sometimes recompute a whole stage?**
  - *A*: If the lost executor held shuffle map outputs, reducers cannot fetch them, so the scheduler may need to rerun the map stage to regenerate those outputs.

### What to Study Next
Open 03_task_retries.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')